# Core clean + xG Dataset - Merge

> **Múltiples ligas**&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuentes:** football-data.co.uk + Understat

## Objetivos

- Aplicar mapping de equipos y ligas, normalizar fechas y temporadas.
- Construir `match_id` compatible y realizar left join.
- Auditar el merge (esperar 100% match).
- Exportar `core_enriched.parquet` (10.660 × 35).

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---|---|
| 0 | Entorno y configuración | Librerías, rutas y configuraciones |
| 1 | Carga de datasets | Core clean y xG raw, verificación de dimensiones |
| 2 | Normalización del dataset xG | Equipos, fechas, ligas y temporadas |
| 3 | Alineación de `match_id` | Generación y verificación de la clave de join |
| 4 | Left join | Merge por `match_id` incorporando `home_xg` y `away_xg` |
| 5 | Auditoría del merge | Cobertura, integridad y consistencia |
| 6 | Conclusiones | Estado final del dataset |
| 7 | Exportación | Guardado del dataset enriquecido en Parquet |


---
##

## 0) Entorno y configuración

Configuración de dependencias, rutas del proyecto y parámetros de referencia.

### 0.1 Imports y rutas

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import json
from IPython.display import display, Markdown

# Rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
RAW_XG_ROOT = PROJECT_ROOT / "data" / "raw" / "xg"
CORE_CLEAN_PATH = PROCESSED_ROOT / "core_multi_league_clean.parquet"
XG_RAW_PATH = RAW_XG_ROOT / "xg_validated.parquet"
ENRICHED_PATH = PROCESSED_ROOT / "core_enriched.parquet"
ENRICHED_SCHEMA_PATH = PROCESSED_ROOT / "core_enriched_schema.json"

# Importación de funciones propias
from src.analysis import normalize_team_name
from src.cleaning import DataValidator

### 0.2 Mapeos de referencia

In [2]:
with open(CONFIG_ROOT / "league_mapping.json") as f:
    LEAGUE_MAP = json.load(f)

with open(CONFIG_ROOT / "team_mapping_xg.json") as f:
    TEAM_MAP = json.load(f)

print(f"League mapping: {len(LEAGUE_MAP)} ligas")
print(f"Team mapping:   {len(TEAM_MAP)} equipos")

League mapping: 3 ligas
Team mapping:   34 equipos


---
##

## 1) Carga de datasets

Lectura del core clean y del dataset xG raw validado.

In [3]:
df_core = pd.read_parquet(CORE_CLEAN_PATH)
df_xg = pd.read_parquet(XG_RAW_PATH)

print(f"Core clean: {len(df_core):,} partidos × {len(df_core.columns)} columnas")
print(f"xG raw:     {len(df_xg):,} partidos × {len(df_xg.columns)} columnas")

assert len(df_core) == 10_660, f"Core: esperados 10,660, obtenidos {len(df_core):,}"
assert len(df_xg) == 10_660, f"xG: esperados 10,660, obtenidos {len(df_xg):,}"
print("\n✓ Ambos datasets con 10,660 filas")

Core clean: 10,660 partidos × 33 columnas
xG raw:     10,660 partidos × 20 columnas

✓ Ambos datasets con 10,660 filas


---
##

## 2) Normalización del dataset xG

Tres transformaciones para alinear el dataset xG con el formato del core antes de construir `match_id`.

### 2.1 Equipos

Aplicación del diccionario de nombres generado en `04_eda_xg`. Los equipos sin entrada en el mapping conservan su nombre original.

In [4]:
print("── Normalización de equipos ──────────────────────────────────────\n")
df_xg["home_team"] = df_xg["home_team"].map(TEAM_MAP).fillna(df_xg["home_team"])
df_xg["away_team"] = df_xg["away_team"].map(TEAM_MAP).fillna(df_xg["away_team"])
examples = list(TEAM_MAP.items())[:3]
print("  Ejemplos de mapping:")

for k, v in examples:
    print(f"   - {k:<20}  →  {v:<20}")

still_unknown = set(df_xg["home_team"].unique()) - set(df_core["HomeTeam"].unique())

if still_unknown:
    print(f"\n  ⚠ Equipos sin correspondencia: {len(still_unknown)}")
    print(f"  Ejemplos: {sorted(still_unknown)[:3]}")
else:
    print("\n  ✓ Mapping completo — todos los equipos tienen correspondencia\n")

print("──────────────────────────────────────────────────────────────────")

── Normalización de equipos ──────────────────────────────────────

  Ejemplos de mapping:
   - Manchester City       →  Man City            
   - Manchester United     →  Man United          
   - Newcastle United      →  Newcastle           

  ✓ Mapping completo — todos los equipos tienen correspondencia

──────────────────────────────────────────────────────────────────


### 2.2 Ligas

Traducción de nombres de liga del formato Understat al formato football-data usando `LEAGUE_MAP`.

In [5]:
print("── Normalización de ligas ───────────────────────────────────────\n")

df_xg["league"] = df_xg["league"].map(LEAGUE_MAP)

for us, core in LEAGUE_MAP.items():
    n = (df_xg["league"] == core).sum()
    print(f"   {us:<22}  →   {core:<12}   {n:>5,} partidos")

print("\n──────────────────────────────────────────────────────────────────")

── Normalización de ligas ───────────────────────────────────────

   ENG-Premier League      →   premier        3,800 partidos
   ESP-La Liga             →   laliga         3,800 partidos
   GER-Bundesliga          →   bundesliga     3,060 partidos

──────────────────────────────────────────────────────────────────


### 2.3 Temporadas

Construir `Season` con el mismo formato que el core (`2015`, `2016`, ...) usando el corte en julio.

In [6]:
SEASON_MAP = {f"{y}{y+1}": str(2000 + y + 1) for y in range(14, 24)}
df_xg["season"] = df_xg["season"].map(SEASON_MAP)

seasons_xg   = sorted(df_xg["season"].unique())
seasons_core = sorted(df_core["Season"].unique())

assert seasons_xg == seasons_core, f"Temporadas no coinciden: {set(seasons_xg) ^ set(seasons_core)}"

print("── Normalización de temporadas ───────────────────────────────────\n")

print(f"   Antes:    {'1415 … 2324':<15}")
print(f"   Después:  {seasons_xg[0]} … {seasons_xg[-1]:<6}   ({len(seasons_xg)} temporadas transformadas)")

print("\n   ✓ Temporadas alineadas con el core dataset")
print("\n──────────────────────────────────────────────────────────────────")

── Normalización de temporadas ───────────────────────────────────

   Antes:    1415 … 2324    
   Después:  2015 … 2024     (10 temporadas transformadas)

   ✓ Temporadas alineadas con el core dataset

──────────────────────────────────────────────────────────────────


---
##

## 3) Alineación de `match_id`

### 3.1 Construcción del identificador

Generación del `match_id` en el dataset xG con el mismo formato que el core: `{League}_{Season}_{HomeTeam}_{AwayTeam}`, con nombres normalizados.

In [7]:
df_xg["match_id"] = (
    df_xg["league"]
    + "_" + df_xg["season"]
    + "_" + df_xg["home_team"].apply(normalize_team_name).str.replace(" ", "_")
    + "_" + df_xg["away_team"].apply(normalize_team_name).str.replace(" ", "_")
)

n_unique = df_xg["match_id"].nunique()

if n_unique == len(df_xg):
    print(f"✓ match_id: {n_unique:,} claves únicas")
else:
    dups = df_xg[df_xg["match_id"].duplicated(keep=False)]
    print(f"⚠ {len(df_xg) - n_unique} match_id duplicados")
    display(dups[["match_id", "date", "league", "home_team", "away_team"]].head(10))

print("\nEjemplos:")
for league in sorted(df_xg["league"].unique()):
    sample = df_xg[df_xg["league"] == league]["match_id"].iloc[0]
    print(f"  • {league}: {sample}")

✓ match_id: 10,660 claves únicas

Ejemplos:
  • bundesliga: bundesliga_2015_bayern_munich_wolfsburg
  • laliga: laliga_2015_almeria_espanol
  • premier: premier_2015_arsenal_crystal_palace


### 3.2 Verificación cruzada

Comparación directa del set de `match_id` entre ambos datasets.

In [8]:
ids_core = set(df_core["match_id"])
ids_xg   = set(df_xg["match_id"])

common       = ids_core & ids_xg
only_in_core = ids_core - ids_xg
only_in_xg   = ids_xg   - ids_core

pct = len(common) / len(ids_core) * 100

bar = "█" * int(pct // 2) + "░" * (50 - int(pct // 2))

print("Cobertura del merge")
print("─────────────────────────────────────────────────────────────")
print(f" {bar} {pct:.2f}%")
print("─────────────────────────────────────────────────────────────\n")

print(f"  {'match_id en core':<18} │ {len(ids_core):>7,}")
print(f"  {'match_id en xG':<18} │ {len(ids_xg):>7,}")
print(f"  {'coincidencias':<18} │ {len(common):>7,}")
print(f"  {'solo en core':<18} │ {len(only_in_core):>7}")
print(f"  {'solo en xG':<18} │ {len(only_in_xg):>7}")

Cobertura del merge
─────────────────────────────────────────────────────────────
 ██████████████████████████████████████████████████ 100.00%
─────────────────────────────────────────────────────────────

  match_id en core   │  10,660
  match_id en xG     │  10,660
  coincidencias      │  10,660
  solo en core       │       0
  solo en xG         │       0


---
##

## 4) Left join

Merge del core clean con las columnas `home_xg` y `away_xg` del dataset xG por `match_id`.

In [9]:
xg_for_merge = df_xg[["match_id", "home_xg", "away_xg"]].copy()

# Convertir xG a float64 estándar
xg_for_merge["home_xg"] = xg_for_merge["home_xg"].astype("float64")
xg_for_merge["away_xg"] = xg_for_merge["away_xg"].astype("float64")

df_enriched = df_core.merge(xg_for_merge, on="match_id", how="left")

print("Resultado del merge")
print("────────────────────────────────────\n")
print(f"  {'Core dataset':<20} │ {df_core.shape[0]:>7,} × {df_core.shape[1]}")
print(f"  {'Dataset enriquecido':<20} │ {df_enriched.shape[0]:>7,} × {df_enriched.shape[1]}")

new_cols = sorted(set(df_enriched.columns) - set(df_core.columns))
print(f"\n  {'Columnas añadidas':<20} │ {', '.join(new_cols)}")

Resultado del merge
────────────────────────────────────

  Core dataset         │  10,660 × 33
  Dataset enriquecido  │  10,660 × 35

  Columnas añadidas    │ away_xg, home_xg


---
##

## 5) Auditoría del merge

Validaciones de integridad sobre el dataset enriquecido.

In [10]:
print("── Auditoría del merge ─────────────────────────────────────────\n")
expected_cols = len(df_core.columns) + len(xg_for_merge.columns) - 1
shape_ok, filas, columnas = DataValidator.get_shape_check(df_enriched, expected_rows=10_660, expected_cols=expected_cols)
nulls_dict = DataValidator.get_nulls_dict(df_enriched[["home_xg", "away_xg"]])
no_nulls = len(nulls_dict) == 0
dups = DataValidator.get_duplicates_count(df_enriched, key_col="match_id")
no_dups = dups == 0
neg_dict = DataValidator.get_negatives_dict(df_enriched[["home_xg", "away_xg"]])
neg = sum(neg_dict.values())
no_neg = neg == 0
max_xg = max(df_enriched["home_xg"].max(), df_enriched["away_xg"].max())
in_range = max_xg <= 10

print(f"  {'✓' if shape_ok else '✗'} {'Shape':<16} │ {filas:,} × {columnas} (esperado: 10,660 × {expected_cols})")
null_txt = f"home_xg={nulls_dict.get('home_xg', 0)}, away_xg={nulls_dict.get('away_xg', 0)}"
print(f"  {'✓' if no_nulls else '✗'} {'Nulos en xG':<16} │ {null_txt}")
print(f"  {'✓' if no_dups else '✗'} {'Duplicados id':<16} │ {dups}")
print(f"  {'✓' if no_neg else '✗'} {'xG negativos':<16} │ {neg}")
print(f"  {'✓' if in_range else '⚠'} {'xG máximo':<16} │ {max_xg:.2f} (umbral: 10)")
print("\n── Goles core vs enriched (cross-check) ────────────────────────\n")
cruzados = DataValidator.get_join_integrity_check(df_core, df_enriched, "League")
for league, stats in cruzados.items():
    match, g_enriched, g_core = stats
    print(f"  [{match}] {league:<15} enriched={g_enriched:>6,}  core={g_core:>6,}")
print("\n───────────────────────────────────────────────────────────────")

all_ok = shape_ok and no_nulls and no_dups and no_neg and in_range
if all_ok:
    print("\n✓ Merge exitoso — dataset listo para exportar")
else:
    print("\n✗ Hay problemas que requieren revisión")

── Auditoría del merge ─────────────────────────────────────────

  ✓ Shape            │ 10,660 × 35 (esperado: 10,660 × 35)
  ✓ Nulos en xG      │ home_xg=0, away_xg=0
  ✓ Duplicados id    │ 0
  ✓ xG negativos     │ 0
  ✓ xG máximo        │ 6.88 (umbral: 10)

── Goles core vs enriched (cross-check) ────────────────────────

  [OK] bundesliga      enriched= 9,234  core= 9,234
  [OK] laliga          enriched= 9,983  core= 9,983
  [OK] premier         enriched=10,614  core=10,614

───────────────────────────────────────────────────────────────

✓ Merge exitoso — dataset listo para exportar


### 5.1 Preview del dataset enriquecido

In [11]:
display(df_enriched[["match_id", "League", "Season", "HomeTeam", "AwayTeam", "home_xg", "away_xg"]].head(5).style
    .format({"home_xg": "{:.2f}", "away_xg": "{:.2f}"})
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th, td", "props": [("text-align", "center")]},
        {"selector": "td:first-child", "props": [("text-align", "left")]},
    ])
)

match_id,League,Season,HomeTeam,AwayTeam,home_xg,away_xg
premier_2015_arsenal_crystal_palace,premier,2015,Arsenal,Crystal Palace,1.55,0.16
premier_2015_leicester_everton,premier,2015,Leicester,Everton,1.28,0.61
premier_2015_man_united_swansea,premier,2015,Man United,Swansea,1.17,0.28
premier_2015_qpr_hull,premier,2015,QPR,Hull,1.90,1.12
premier_2015_stoke_aston_villa,premier,2015,Stoke,Aston Villa,0.42,0.91


---
##

## 6) Conclusiones

El dataset `core_enriched` integra los **10.660 partidos** del core clean con las métricas de Expected Goals de Understat, resultando en un dataset de **10.660 × 35 columnas**.

### 6.1 Normalización

Se aplicaron tres transformaciones al dataset xG para alinear su nomenclatura con el core:

| Aspecto | Transformación |
|---------|---------------|
| Equipos | Nombres mapeados vía `team_mapping_xg.json` |
| Ligas | Understat → football-data (`ENG-Premier League` → `premier`, etc.) |
| Temporadas | Formato interno → año de cierre (`1415` → `2015`, etc.) |

### 6.2 Resultado del merge

- **Cobertura:** 100 % — los 10.660 `match_id` del core tienen correspondencia exacta en el dataset xG.
- **Columnas incorporadas:** `home_xg` y `away_xg`.
- **Sin nulos, duplicados ni negativos** en las columnas añadidas.
- **Goles cruzados:** coincidencia exacta entre ambas fuentes en las tres ligas.

### 6.3 Dataset final

| Métrica | Valor |
|---------|-------|
| Partidos | 10.660 |
| Columnas | 35 (33 core + 2 xG) |
| Ligas | Bundesliga, La Liga, Premier League |
| Temporadas | 2014-15 → 2023-24 |
| Formato | `core_enriched.parquet` + `core_enriched_schema.json` |

El dataset está listo para la fase de análisis.

---
##

## 7) Exportación

Guardado del dataset enriquecido en formato Parquet con esquema JSON de referencia.

In [12]:
df_enriched.to_parquet(ENRICHED_PATH, index=False)

schema = {
    "num_rows": len(df_enriched),
    "num_columns": len(df_enriched.columns),
    "num_leagues": df_enriched["League"].nunique(),
    "leagues": sorted(df_enriched["League"].unique().tolist()),
    "columns": sorted(df_enriched.columns.tolist()),
    "dtypes": {col: str(df_enriched[col].dtype) for col in sorted(df_enriched.columns)},
    "matches_per_league": df_enriched.groupby("League").size().to_dict(),
    "xg_source": "understat (vía soccerdata)"
}

with open(ENRICHED_SCHEMA_PATH, "w") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print(f"Dataset enriched exportado:")
print(f"  {len(df_enriched):,} filas × {len(df_enriched.columns)} columnas\n")

print(f"Archivos guardados:")
print(f"  · Dataset → {ENRICHED_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {ENRICHED_SCHEMA_PATH.relative_to(PROJECT_ROOT)}")

Dataset enriched exportado:
  10,660 filas × 35 columnas

Archivos guardados:
  · Dataset → data/processed/core_enriched.parquet
  · Esquema → data/processed/core_enriched_schema.json


---
##